# Model NER

### 1. Download Libraries

In [ ]:

!pip install transformers torch -q


### 2. Load txt file

In [ ]:
TXT_PATH = r"/content/FR001400QV82_AVMAFC_30Jun2028.txt"

with open(TXT_PATH, encoding="utf-8") as f:
    text = f.read()

print("=== RAW CHAT MESSAGE ===")
print(text)

=== RAW CHAT MESSAGE ===
11:49:05 I'll revert regarding BANK ABC to try to do another 200 mio at 2Y
FR001400QV82	AVMAFC FLOAT	06/30/28
offer 2Y EVG estr+45bps
estr average Estr average / Quarterly interest payment




### 3. Load and Run BERT NER Model

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

MODEL_NAME = "dslim/bert-base-NER"

print(f"Loading model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)

ner_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",  # merges B-ORG + I-ORG tokens into one span
    device=-1,                       # -1 = CPU (no GPU needed for short texts)
)

print("Model loaded successfully!")

Loading model: dslim/bert-base-NER


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully!


In [ ]:
# Run the model on our chat text
model_results = ner_pipeline(text)

print("=== MODEL NER RESULTS ===")
print(f"{'Label':<12} {'Text':<30} {'Score':>7}  Position")
print("-" * 60)
for r in model_results:
    print(f"{r['entity_group']:<12} {repr(r['word']):<30} {r['score']:.4f}  [{r['start']}:{r['end']}]")

=== MODEL NER RESULTS ===
Label        Text                             Score  Position
------------------------------------------------------------
ORG          'BANK ABC'                     0.8065  [31:39]
ORG          'A'                            0.7277  [88:89]
ORG          '##VMAFC'                      0.8472  [89:94]


In [ ]:
def run_ner_model(text):
    try:
        from transformers import pipeline
        ner = ner_pipeline
        results = ner(text)
        entities = {}
        for r in results:
            if r["entity_group"] == "ORG" and "Counterparty" not in entities:
                entities["Counterparty"] = r["word"]
        return entities
    except ImportError:
        print("  [WARNING] transformers not installed — Counterparty will be null")
        return {}

print("\n=== NER Model ===")
model_entities = run_ner_model(text)
for k, v in model_entities.items():
    print(f"  {k:<20}  '{v}'")



=== NER Model ===
  Counterparty          'BANK ABC'


### Extra: Extracting required entities using regex

In [ ]:
import re
# Target entities for the chat document
TARGET_ENTITIES = [
    "Counterparty",
    "Notional",
    "ISIN",
    "Underlying",
    "Maturity",
    "Bid",
    "Offer",
    "PaymentFrequency",
]

# Regex rules — one pattern per entity
RULES = [
    ("ISIN",             re.compile(r"\b([A-Z]{2}[A-Z0-9]{10})\b")),
    ("Notional",         re.compile(r"\b(\d+[\d\.,]*\s*(?:mio|million|bn|billion))\b", re.IGNORECASE)),
    ("Maturity",         re.compile(r"\b(\d+[Yy]\s+EVG|\d+[Yy])\b")),
    ("Bid",              re.compile(r"\boffer\s+\d+[Yy](?:\s+EVG)?\s+((?:estr|euribor|libor|sofr|sonia)[+\-]\d+\.?\d*\s*bps?)", re.IGNORECASE)),
    ("Offer",            re.compile(r"\boffer\s+((?:estr|euribor|libor|sofr|sonia)[+\-]\d+\.?\d*\s*bps?)\b", re.IGNORECASE)),
    ("PaymentFrequency", re.compile(r"\b(monthly|quarterly|semi[-\s]?annual|annual)\b", re.IGNORECASE)),
    ("Underlying",       re.compile(r"\b([A-Z]{2,6}\s+(?:FLOAT|FIXED|SWAP|CAP|FLOOR)\s+\d{2}/\d{2}/\d{2,4})\b")),
]

In [ ]:
def run_regex(text):
    entities = {}
    for label, pattern in RULES:
        match = pattern.search(text)
        if match:
            entities[label] = match.group(1).strip()
    return entities

print("=== Regex ===")
regex_entities = run_regex(text)
for k, v in regex_entities.items():
    print(f"  {k:<20} → '{v}'")


=== Regex ===
  ISIN                 → 'FR001400QV82'
  Notional             → '200 mio'
  Maturity             → '2Y'
  Bid                  → 'estr+45bps'
  PaymentFrequency     → 'Quarterly'
  Underlying           → 'AVMAFC FLOAT	06/30/28'


In [ ]:
import json
def build_output(regex_entities, model_entities, txt_path):
    # Merge: model fills what regex can't catch
    merged = {**regex_entities, **model_entities}

    # Build final output — null for anything not found
    entities = {target: merged.get(target, None) for target in TARGET_ENTITIES}

    return {
        "source":        txt_path.split("/")[-1],
        "engine":        "bert_ner + regex_rules",
        "document_type": "chat_txt",
        "entities":      entities,
    }

print("\n=== Final Output ===")
output = build_output(regex_entities, model_entities, TXT_PATH)
print(json.dumps(output, indent=2))


=== Final Output ===
{
  "source": "FR001400QV82_AVMAFC_30Jun2028.txt",
  "engine": "bert_ner + regex_rules",
  "document_type": "chat_txt",
  "entities": {
    "Counterparty": "BANK ABC",
    "Notional": "200 mio",
    "ISIN": "FR001400QV82",
    "Underlying": "AVMAFC FLOAT\t06/30/28",
    "Maturity": "2Y",
    "Bid": "estr+45bps",
    "Offer": null,
    "PaymentFrequency": "Quarterly"
  }
}
